# ShopFlow A/B Test Analysis

This notebook demonstrates fixed-horizon conversion analysis on **simulated** experiment counts. Replace the illustrative counts with a validated experiment extract before making a real product decision. Do not interpret synthetic evidence as causal evidence about real customers.

## Design checklist
- Define one primary metric and a meaningful minimum detectable effect (MDE).
- Randomize eligible users and analyze by assigned group (intention-to-treat).
- Validate assignment balance, exposure, duplicate users, and guardrail metrics.
- Fix sample size and stopping date before looking at significance; avoid repeated peeking.
- Pre-specify subgroup analyses and adjust for multiple comparisons when needed.

In [ ]:
import sys
from pathlib import Path

# Make repository imports work when Jupyter starts from the notebooks/ directory.
search_from = Path.cwd().resolve()
repo_root = next(
    (candidate for candidate in (search_from, *search_from.parents)
     if (candidate / 'src' / 'experimentation.py').is_file()),
    None,
)
if repo_root is None:
    raise FileNotFoundError('Could not locate repository root containing src/experimentation.py')
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from src.experimentation import (
    difference_confidence_interval,
    interpret_result,
    required_sample_size_per_group,
    two_proportion_z_test,
)

## 1. Set the experiment design
The example uses a 34% baseline and a +2 percentage-point absolute MDE. Alpha is 5% (two-sided) and target power is 80%. The MDE is absolute, not a relative percentage.

In [ ]:
baseline_rate = 0.34
mde = 0.02
alpha = 0.05
target_power = 0.80
n_per_group = required_sample_size_per_group(baseline_rate, mde, alpha, target_power)
print(f"Approximate required sample size per group: {n_per_group:,}")
print(f"Total required sample size: {2*n_per_group:,}")

This is a normal-approximation planning calculation, not a substitute for checking assumptions, traffic allocation, expected attrition, clustering, or the actual outcome definition.

## 2. Analyze illustrative fixed-horizon results
The counts below are teaching examples, not a claim about the live ShopFlow dataset or real users.

In [ ]:
control_successes, control_total = 340, 1000
treatment_successes, treatment_total = 410, 1000
result = two_proportion_z_test(control_successes, control_total, treatment_successes, treatment_total)
ci_low, ci_high = difference_confidence_interval(
    control_successes, control_total, treatment_successes, treatment_total, 0.95
)
print(f"Control conversion: {result['control_rate']:.1%}")
print(f"Treatment conversion: {result['treatment_rate']:.1%}")
print(f"Absolute lift: {result['absolute_lift']:.1%} ({result['absolute_lift']*100:.1f} percentage points)")
print(f"Relative lift: {result['relative_lift']:.1%}")
print(f"95% CI for absolute lift: [{ci_low:.1%}, {ci_high:.1%}]")
print(f"Two-sided p-value: {result['p_value']:.6g}")
print(interpret_result(result['p_value'], alpha))

### Practical significance and decision interpretation

The minimum detectable effect (MDE) is the smallest absolute lift the experiment was designed to detect under its planning assumptions. It is not automatically the business's rollout threshold.

Compare the confidence interval with the pre-specified MDE:

- If the lower confidence bound exceeds the MDE, the interval supports an effect greater than the target.
- If the interval overlaps the MDE, the evidence does not clearly establish whether the effect exceeds that target.
- If the upper confidence bound is below the MDE, the interval does not support an effect as large as the target.

These interpretations assume the experiment design, randomization, outcome definition, and statistical model are valid. This notebook uses simulated counts for teaching, not real customer evidence.

In [ ]:
mde = 0.02  # Absolute effect; two percentage points

if ci_low > mde:
    practical_interpretation = (
        "The 95% confidence interval lies entirely above the illustrative MDE."
    )
elif ci_high < mde:
    practical_interpretation = (
        "The 95% confidence interval lies entirely below the illustrative MDE."
    )
else:
    practical_interpretation = (
        "The 95% confidence interval overlaps the illustrative MDE."
    )

print(f"Illustrative MDE: {mde * 100:.2f} percentage points")
print(f"95% CI: [{ci_low * 100:.2f}, {ci_high * 100:.2f}] percentage points")
print(practical_interpretation)
print("Decision note: validate experiment assumptions and guardrails before rollout.")

## 3. Interpretation and decision guardrails
- Statistical significance does not guarantee a meaningful business impact. Compare the confidence interval with the pre-defined MDE and rollout costs.
- Check guardrails such as refunds, errors, page latency, cancellations, and revenue per eligible user. Do not select a winner if a critical guardrail deteriorates.
- If users can generate multiple events, aggregate to the randomized unit before testing. Event-level rows are not independent users.
- If the same user can appear in both groups, assignment is invalid; deduplicate and investigate the assignment pipeline.
- Do not repeatedly test and stop when p < 0.05 unless using a valid sequential-testing design.
- For many variants or outcomes, pre-specify the primary comparison and control family-wise error or false discovery rate.
- Novelty effects and selection bias require design/context checks; a p-value does not detect them.
- Regression can improve precision when covariates are pre-treatment and the model is specified appropriately. It cannot repair broken randomization or post-treatment adjustment.

## 4. Final recommendation template
1. **Decision:** ship, do not ship, or continue collecting data under the pre-specified plan.
2. **Evidence:** conversion in each group, absolute/relative lift, confidence interval, p-value, and sample size.
3. **Practicality:** expected incremental conversions and operational cost at intended traffic.
4. **Guardrails:** summarize each guardrail against its pre-defined tolerance.
5. **Validity:** report sample-ratio checks, exposure quality, missingness, peeking, multiplicity, and subgroup limitations.
6. **Caveat:** this notebook's built-in examples are simulated and do not support a real-world causal claim.